In [3]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

In [4]:
# --- 1. Gerar Dados de Treinamento ---
np.random.seed(42)
tf.random.set_seed(42)

# Gerar muitos pontos para filtrar os problemáticos
num_samples = 5000
angles_all = np.random.uniform(0, 3 * np.pi, num_samples).reshape(-1, 1)
tan_values_all = np.tan(angles_all)

# Remover pontos com valores extremos (assíntotas)
mask = np.abs(tan_values_all) < 10
angles_train = angles_all[mask]
tan_values_train = tan_values_all[mask]

# Adicionar ruído gaussiano
noise = np.random.normal(0, 0.05, tan_values_train.shape)
tan_values_train += noise

In [5]:
# --- 2. Definir o Modelo ---
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(1,)),
    tf.keras.layers.Dense(50, activation='tanh'),
    tf.keras.layers.Dense(50, activation='tanh'),
    tf.keras.layers.Dense(50, activation='tanh'),
    tf.keras.layers.Dense(1)
])

model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
              loss='mse')


In [6]:
# --- 3. Treinar o Modelo ---
history = model.fit(angles_train, tan_values_train,
                    epochs=1000,
                    batch_size=32,
                    verbose=0)


In [7]:
# --- 4. Dados de Teste ---
angles_test = np.linspace(0, 3 * np.pi, 500).reshape(-1, 1)
tan_values_true = np.tan(angles_test)

In [ ]:
# Também filtramos para visualizar apenas parte segura
mask_test = np.abs(tan_values_true) < 10
angles_test = angles_test[mask_test]
tan_values_true = tan_values_true[mask_test]
# --- 5. Prever ---
tan_values_predicted = model.predict(angles_test)


15/15 [==============================] - 0s 997us/step


: 

In [ ]:
# --- 6. Visualizar ---
plt.figure(figsize=(10, 6))
plt.scatter(angles_train, tan_values_train, label='Dados de Treinamento (ruído)', alpha=0.4)
plt.plot(angles_test, tan_values_true, label='tan(x) verdadeiro', color='blue')
plt.plot(angles_test, tan_values_predicted, label='tan(x) previsto', color='red')
plt.xlabel('Ângulo (radianos)')
plt.ylabel('tan(x)')
plt.title('Interpolação da Função Tangente com TensorFlow')
plt.legend()
plt.grid(True)
plt.ylim(-10, 10)
plt.show()

In [ ]:
# --- 7. Erro ---
mse = tf.reduce_mean(tf.square(tan_values_true - tan_values_predicted))
print(f"Erro Quadrático Médio nos Dados de Teste: {mse.numpy():.6f}")
